In [ ]:
pip install pyvi

In [ ]:
import pandas as pd


DATA = "/content/data.csv"
data = pd.read_csv(DATA, index_col=False)

# Kiểm tra các dòng trùng lặp
duplicates = data[data.duplicated(keep=False)]

# In các dòng trùng lặp
print("Các dòng trùng lặp:")
print(duplicates)

# Đếm số lượng dòng trùng lặp
num_duplicates = data.duplicated().sum()
print(f"Số lượng dòng trùng lặp: {num_duplicates}")

Các dòng trùng lặp:
                                                 content label  start
1                                              Tuyệt vời   POS      5
4                                       Vải đẹp, dày dặn   POS      5
8                            Đóng gói sản phẩm chắc chắn   POS      5
10                                  Shop phục vụ rất tốt   POS      5
13                         Thời gian giao hàng rất nhanh   POS      5
...                                                  ...   ...    ...
31447                                                NaN   POS      5
31451  Chất lượng sản phẩm tuyệt vời Đóng gói sản phẩ...   POS      5
31453                                    Giao hàng nhanh   POS      5
31455                                    Không đáng tiền   NEG      1
31456                                       Quần rất đẹp   POS      5

[5125 rows x 3 columns]
Số lượng dòng trùng lặp: 4259


In [ ]:
# Xóa các dòng trùng lặp
data = data.drop_duplicates()

# In DataFrame sau khi xóa dòng trùng lặp
print("DataFrame sau khi xóa các dòng trùng lặp:")
print(data)

DataFrame sau khi xóa các dòng trùng lặp:
                                                 content label  start
0                                          Áo bao đẹp ạ!   POS      5
1                                              Tuyệt vời   POS      5
2                              2day ao khong giong trong   NEG      1
3                             Mùi thơm,bôi lên da mềm da   POS      5
4                                       Vải đẹp, dày dặn   POS      5
...                                                  ...   ...    ...
31452                     Chất tốt, Shop phục vụ rất tốt   POS      5
31454  Hàng y hình, đóng gói ko cẩn thận nên quả ngực...   NEU      3
31457                             Hàng đẹp đúng giá tiền   POS      5
31458                                    Chất vải khá ổn   POS      4
31459  áo rất ok nhé , vải mịn , len cao cổ này phối ...   POS      5

[27201 rows x 3 columns]


In [ ]:
# Đếm số hàng của mỗi giá trị trong cột 'label'
label_counts = data['label'].value_counts()

# In kết quả
print("Số hàng của mỗi giá trị trong cột 'label':")
print(label_counts)

Số hàng của mỗi giá trị trong cột 'label':
label
NEG    16061
NEU    14372
Name: count, dtype: int64


In [ ]:
#lấy ra label muốn tăng cường
DATA_HATE = '/content/augmentation_label_POS.csv'

data = pd.read_csv(DATA, index_col=False)
data.dropna(inplace=True)

label1 = data.loc[data['label']=='POS']
label1.drop_duplicates(inplace=True)
print(label1)
label1.to_csv(DATA_HATE, header=False, index=False, sep="|")

                                                 content label  start
0                                          Áo bao đẹp ạ!   POS      5
1                                              Tuyệt vời   POS      5
3                             Mùi thơm,bôi lên da mềm da   POS      5
4                                       Vải đẹp, dày dặn   POS      5
5                         Hàng rất đẹp, rất chi là ưng ý   POS      5
...                                                  ...   ...    ...
31450  Chất lượng sản phẩm phù hợp vs giá tiền, vải m...   POS      5
31452                     Chất tốt, Shop phục vụ rất tốt   POS      5
31457                             Hàng đẹp đúng giá tiền   POS      5
31458                                    Chất vải khá ổn   POS      4
31459  áo rất ok nhé , vải mịn , len cao cổ này phối ...   POS      5

[16428 rows x 3 columns]


<ipython-input-5-03332a252a75>:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  label1.drop_duplicates(inplace=True)


# Tăng cường văn bản bằng kỹ thuật EDA

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

class Argument:
    input = "/content/augmentation_label_NEU.csv"
    output = "drive/My Drive/augmentation_data_NEU.txt"
    num_aug = 3 #số lượng các bản sao (augmented data) mà bạn muốn tạo ra từ dữ liệu gốc
    alpha = 0.15 #hỉ định mức độ thay đổi nhỏ trong việc tạo ra dữ liệu mới từ dữ liệu gốc


args = Argument()

In [ ]:

import random
from random import shuffle

random.seed(1)
import json


# stop words
stop_words = []
with open("/content/vietnamese-stopwords.txt", "r") as f:
    stop_words = []
    for line in f:
        dd = line.strip('\n')
        stop_words.append(dd)


import re



from nltk.corpus import wordnet
########################################################################
#Synonym Replacement
# thay thế một số từ trong câu bằng từ đồng nghĩa của chúng.
# hàm get_synonyms sẽ tìm các từ đồng nghĩa từ một tệp JSON chứa thông tin về từ vựng tiếng Việt
########################################################################
def synonym_replacement(words, n):
    new_words = words.copy() #Tạo bản sao
    random_word_list = list(set([word for word in words if word not in stop_words]))#Lọc bỏ stopwword
    random.shuffle(random_word_list) #Trộn ngẫu nhiên danh sách các từ.
    num_replaced = 0
    for random_word in random_word_list:
        synonyms = get_synonyms(random_word) #Lấy các từ đồng nghĩa cho mỗi từ từ ngẫu nhiên
        if len(synonyms) >= 1:
            synonym = random.choice(list(synonyms))
            new_words = [synonym if word == random_word else word for word in new_words]#Tạo lại câu mới bằng cách thay thế các từ trong câu gốc bằng từ đồng nghĩa
            # print("replaced", random_word, "with", synonym)
            num_replaced += 1
        if num_replaced >= n:  # chỉ thay thế tối đa n từ
            break


    sentence = ' '.join(new_words)
    new_words = sentence.split(' ')

    return new_words



#Tìm các từ đồng nghĩa cho một từ trong từ điển word_net_vi.json
def get_synonyms(word):
    synonyms = set()
    with open("/content/word_net_vi.json", "r") as f:
        wordnet = json.load(f)

    for key, value in wordnet.items():
        if key.strip() == word:
            for v in value:
                synonyms.add(v.strip())

        if word in synonyms:
            synonyms.remove(word)
    return list(synonyms)


########################################################################
# Random deletion
# Xóa ngẫu nhiên các từ trong câu với xác suất p
# Nếu tất cả các từ bị xóa hết, một từ ngẫu nhiên sẽ được giữ lại.
########################################################################

def random_deletion(words, p):
    #  nếu chỉ có một từ thì đừng xóa nó
    if len(words) == 1:
        return words

    # xóa ngẫu nhiên các từ có xác suất p
    new_words = []
    for word in words:
        r = random.uniform(0, 1)
        if r > p:
            new_words.append(word)

    # nếu xóa hết tất cả các từ, chỉ cần trả về một từ ngẫu nhiên
    if len(new_words) == 0:
        rand_int = random.randint(0, len(words) - 1)
        return [words[rand_int]]

    return new_words


########################################################################
# Random swap
# Hoán đổi ngẫu nhiên hai từ trong câu n lần
########################################################################

def random_swap(words, n): #Hoán đổi ngẫu nhiên các từ trong câu
    new_words = words.copy()
    for _ in range(n):
        if len(new_words) > 0:
            new_words = swap_word(new_words)
    return new_words


def swap_word(new_words):#Hoán đổi hai từ ngẫu nhiên trong câu
    random_idx_1 = random.randint(0, len(new_words) - 1)
    random_idx_2 = random_idx_1
    counter = 0
    while random_idx_2 == random_idx_1:
        random_idx_2 = random.randint(0, len(new_words) - 1)
        counter += 1
        if counter > 3:
            return new_words
    new_words[random_idx_1], new_words[random_idx_2] = new_words[random_idx_2], new_words[random_idx_1]
    return new_words


########################################################################
# Random insertion
# Chèn ngẫu nhiên n từ vào câu
########################################################################

def random_insertion(words, n):#Chèn một từ đồng nghĩa vào trong câu
    new_words = words.copy()
    for _ in range(n):
        add_word(new_words)
    return new_words


def add_word(new_words): #Chọn ngẫu nhiên một từ trong câu và thêm một từ đồng nghĩa vào vị trí ngẫu nhiên trong câu
    synonyms = []
    counter = 0
    while len(synonyms) < 1 and len(new_words) > 0:
    # while len(synonyms) < 1:
        random_word = new_words[random.randint(0, len(new_words) - 1)]
        synonyms = get_synonyms(random_word)
        counter += 1
        if counter >= 10:
            return

    if len(new_words) > 0:
        random_synonym = synonyms[0]
        random_idx = random.randint(0, len(new_words) - 1)
        new_words.insert(random_idx, random_synonym)


########################################################################
# hàm chính
########################################################################

def eda(sentence, alpha_sr=0.1, alpha_ri=0.1, alpha_rs=0.1, p_rd=0.1, num_aug=9):
    sentence = get_only_chars(sentence)
    words = sentence.split(' ')
    words = [word for word in words if word != '']
    num_words = len(words)

    augmented_sentences = []

    if len(words) <= 0:
        return augmented_sentences
    num_new_per_technique = int(num_aug / 4) + 1
    n_sr = max(1, int(alpha_sr * num_words))
    n_ri = max(1, int(alpha_ri * num_words))
    n_rs = max(1, int(alpha_rs * num_words))

    # sr
    for _ in range(num_new_per_technique):
        a_words = synonym_replacement(words, n_sr)
        augmented_sentences.append(' '.join(a_words))

    # ri
    for _ in range(num_new_per_technique):
        a_words = random_insertion(words, n_ri)
        augmented_sentences.append(' '.join(a_words))

    # rs
    for _ in range(num_new_per_technique):
        a_words = random_swap(words, n_rs)
        augmented_sentences.append(' '.join(a_words))

    # rd
    for _ in range(num_new_per_technique):
        a_words = random_deletion(words, p_rd)
        augmented_sentences.append(' '.join(a_words))

    augmented_sentences = list(set(augmented_sentences))
    augmented_sentences = [get_only_chars(sentence) for sentence in augmented_sentences]
    shuffle(augmented_sentences)

    # cắt bớt để chúng ta có số lượng câu tăng cường như mong muốn
    if num_aug >= 1:
        augmented_sentences = augmented_sentences[:num_aug]
    else:
        keep_prob = num_aug / len(augmented_sentences)
        augmented_sentences = [s for s in augmented_sentences if random.uniform(0, 1) < keep_prob]

    # nối thêm câu gốc
    augmented_sentences.append(sentence)

    return augmented_sentences


# output file
output = None
if args.output:
    output = args.output
else:
    from os.path import dirname, basename, join

    output = join(dirname(args.input), 'eda_' + basename(args.input))

#số câu tăng cường cần tạo cho mỗi câu gốc
num_aug =3  # default
if args.num_aug:
    num_aug = args.num_aug

# mỗi câu phải đổi bao nhiêu
alpha = 0.1  # default
if args.alpha:
    alpha = args.alpha


# đọc dữ liệu từ file train_orig và áp dụng augmentation cho từng câu văn trong đó
def gen_eda(train_orig, output_file, alpha, num_aug=9):
    try:
        writer = open(output_file, 'w')
        lines = open(train_orig, 'r').readlines() #Đọc toàn bộ nội dung file

        writer.write("content,label,start\n")  # Viết tiêu đề
        augm = ""
        for i, line in enumerate(lines):
            try:
                parts = line[:-1].split('|')
                label = parts[1]
                sentence = parts[0]
                aug_sentences = eda(sentence, alpha_sr=alpha, alpha_ri=alpha, alpha_rs=alpha, p_rd=alpha, num_aug=num_aug) #Áp dụng augmentation cho mỗi câu
                for aug_sentence in aug_sentences:
                    augm += f"{aug_sentence},{label},0\n"  # Ghi vào tệp với định dạng cụ thể
            except Exception as e:
                print(f"Error processing line {i}: {e}")
                print(parts)
                pass

        if len(augm) > 0:
            writer.write(augm)
        else:
            print("No augmented sentences generated.")

        writer.close()
        print(f"Generated augmented sentences with EDA for {train_orig} to {output_file} with num_aug={num_aug}")
    except Exception as e:
        print(f"Failed to generate EDA: {e}")



## Main code

In [ ]:
if __name__ == "__main__":
    # tạo văn bản mới
    gen_eda(args.input, args.output, alpha=alpha, num_aug=num_aug)

Generated augmented sentences with EDA for /content/augmentation_label_NEU.csv to drive/My Drive/augmentation_data_NEU.txt with num_aug=3


# Kết quả và nối với bản gốc

In [ ]:

DATA_HATE = '/content/augmentation_label_POS.csv'

data = pd.read_csv('/content/data.csv', index_col=False)
data.dropna(inplace=True)

label1 = data.loc[data['label']=='POS']
label1.drop_duplicates(inplace=True)
print(label1)
label1.to_csv(DATA_HATE, header=True, index=False)

                                                 content label  start
0                                          Áo bao đẹp ạ!   POS      5
1                                              Tuyệt vời   POS      5
3                             Mùi thơm,bôi lên da mềm da   POS      5
4                                       Vải đẹp, dày dặn   POS      5
5                         Hàng rất đẹp, rất chi là ưng ý   POS      5
...                                                  ...   ...    ...
31450  Chất lượng sản phẩm phù hợp vs giá tiền, vải m...   POS      5
31452                     Chất tốt, Shop phục vụ rất tốt   POS      5
31457                             Hàng đẹp đúng giá tiền   POS      5
31458                                    Chất vải khá ổn   POS      4
31459  áo rất ok nhé , vải mịn , len cao cổ này phối ...   POS      5

[16428 rows x 3 columns]


<ipython-input-31-4f80d147beac>:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  label1.drop_duplicates(inplace=True)


In [ ]:
# Extra augmentation
import pandas as pd

DATA = '/content/augmentation_label_POS.csv'

DATA_AUG = '/content/augmentation_data_NEU.txt'
DATA_AUG_2 = '/content/augmentation_data_NEG.txt'

DATA_AUG_FINAL = '/content/data_final_3.csv'

data_hate = pd.read_csv(DATA_AUG, index_col=False, on_bad_lines='skip')
data_hate_2 = pd.read_csv(DATA_AUG_2, index_col=False, on_bad_lines='skip')

data_hate_final = pd.concat([data_hate, data_hate_2])
data_hate_final.drop_duplicates(subset ="content", keep = False, inplace = True)

data = pd.read_csv(DATA, index_col=False)
data_aug = pd.concat([data, data_hate_final])

data_aug.to_csv(DATA_AUG_FINAL, index=False)

In [ ]:
data33=pd.read_csv('/content/data_final_3.csv')

In [ ]:
# Đếm số hàng của mỗi giá trị trong cột 'label'
label_counts = data33['label'].value_counts()

# In kết quả
print("Số hàng của mỗi giá trị trong cột 'label':")
print(label_counts)

Số hàng của mỗi giá trị trong cột 'label':
label
POS    16428
NEG    16061
NEU    14372
Name: count, dtype: int64
